# Multi-Broker MQTT Publisher

This notebook demonstrates publishing messages to local and public MQTT brokers.

## Setup

To use multiple brokers, edit `config.yaml` and change the `mqtt` section:

```yaml
mqtt:
  active_profiles: [local, mqtthq]  # or [local] or [mqtthq]
  profiles:
    local:
      host: "127.0.0.1"
      port: 1883
      tls: false
    mqtthq:
      host: "broker.mqttdashboard.com"
      port: 1883
      tls: false
```

## Imports

In [ ]:
import json
import time
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher

## Load Configuration

Load the configuration from `config.yaml`. This demonstrates how the new multi-broker support works:

In [ ]:
# Load configuration
cfg = load_config()

# Primary broker (first in active_profiles)
print(f"Primary broker: {cfg.mqtt.host}:{cfg.mqtt.port}")

# All available brokers
print(f"\nAvailable brokers:")
for profile_name, mqtt_cfg in cfg.mqtt_configs.items():
    print(f"  - {profile_name}: {mqtt_cfg.host}:{mqtt_cfg.port}")

---

# Option 1: Publish to Primary Broker Only

This simple approach publishes to the primary (first) broker. Works with both single and multiple broker configs.

In [ ]:
# Connect to primary broker
connector = MqttConnector(cfg.mqtt, client_id_suffix="publisher-primary")
connector.connect()

# Wait for connection
if connector.wait_for_connection(timeout=5.0):
    publisher = MqttPublisher(connector)
    
    # Publish a test message
    topic = f"{cfg.mqtt.base_topic}/publisher-demo/primary"
    payload = json.dumps({"message": "Hello from primary broker", "timestamp": time.time()})
    
    result = publisher.publish_json(topic, payload)
    if result.rc == 0:
        print(f"✓ Published to {cfg.mqtt.host}:{cfg.mqtt.port}")
        print(f"  Topic: {topic}")
        print(f"  Payload: {payload}")
    else:
        print(f"✗ Failed to publish (code: {result.rc})")
else:
    print(f"✗ Failed to connect to {cfg.mqtt.host}:{cfg.mqtt.port}")

In [ ]:
# Clean up
connector.disconnect()

---

# Option 2: Publish to a Specific Named Broker

Choose which broker to publish to by name. This works when using multiple brokers. to se the name of the availabul broksers see the output of the load configuation cell

In [ ]:
# Pick a specific broker by name
broker_name = "hivemq_cloud"  # use "local" for local broker and "hivemq_cloud" to use the public broker

if broker_name not in cfg.mqtt_configs:
    print(f"✗ Broker '{broker_name}' not available. Available: {list(cfg.mqtt_configs.keys())}")
else:
    broker_cfg = cfg.mqtt_configs[broker_name]
    
    # Connect to selected broker
    connector = MqttConnector(broker_cfg, client_id_suffix=f"publisher-{broker_name}")
    connector.connect()
    
    if connector.wait_for_connection(timeout=5.0):
        publisher = MqttPublisher(connector)
        
        # Publish a test message
        topic = f"{broker_cfg.base_topic}/publisher-demo/{broker_name}"
        payload = json.dumps({"message": f"Hello from {broker_name} broker", "timestamp": time.time()})
        
        result = publisher.publish_json(topic, payload)
        if result.rc == 0:
            print(f"✓ Published to {broker_name}")
            print(f"  Host: {broker_cfg.host}:{broker_cfg.port}")
            print(f"  Topic: {topic}")
            print(f"  Payload: {payload}")
        else:
            print(f"✗ Failed to publish (code: {result.rc})")
    else:
        print(f"✗ Failed to connect to {broker_cfg.host}:{broker_cfg.port}")

In [ ]:
# Clean up
connector.disconnect()

---

# Option 3: Route Messages to Different Brokers

Send different types of messages to different brokers:
- **Local broker**: Internal process communication (faster, private)
- **Public broker**: Share status information (external visibility)

## How it works

This example demonstrates a practical message routing pattern:

1. **Connect to all configured brokers** - Establish connections to local and public brokers
2. **Route internal messages** - Send heartbeats, sensor readings, and debug logs to the local broker only
   - These messages are for internal system communication
   - Kept private on your local network
   - No external visibility
   
3. **Route status messages** - Send system status and metrics to public brokers
   - These messages are for external sharing
   - Show overall system health and performance
   - Can be monitored by external dashboards or services

This separation allows you to:
- Keep sensitive internal details private
- Share only relevant status information publicly
- Optimize bandwidth by using local broker for high-frequency internal messages
- Use public broker only for periodic status updates

In [ ]:
# Create and connect to all available brokers
connectors = {}
publishers = {}
failed_brokers = []

print("Connecting to all available brokers...\n")

for profile_name, broker_cfg in cfg.mqtt_configs.items():
    print(f"Connecting to {profile_name}...")
    connector = MqttConnector(broker_cfg, client_id_suffix=f"publisher-{profile_name}")
    connector.connect()
    
    # Wait longer for cloud brokers (they may need more time for TLS handshake)
    timeout = 10.0 if broker_cfg.tls else 5.0
    
    if connector.wait_for_connection(timeout=timeout):
        connectors[profile_name] = connector
        publishers[profile_name] = MqttPublisher(connector)
        print(f"  ✓ Connected to {profile_name} ({broker_cfg.host}:{broker_cfg.port})\n")
        # Small delay before next connection to avoid conflicts
        time.sleep(0.5)
    else:
        print(f"  ✗ Failed to connect to {profile_name} ({broker_cfg.host}:{broker_cfg.port})")
        failed_brokers.append(profile_name)
        connector.disconnect()
        print()

if failed_brokers:
    print(f"Note: Failed to connect to {failed_brokers}. Check credentials or remove from config.\n")

print(f"Ready to publish to {len(publishers)} broker(s): {list(publishers.keys())}")

# Give connections time to stabilize
if publishers:
    print("Waiting for connections to stabilize...")
    time.sleep(2)
    print("Ready!\n")

In [ ]:
# Route different message types to different brokers
if len(publishers) == 0:
    print("No brokers connected. Cannot publish.")
else:
    print("Routing messages based on type:\n")
    
    # Internal messages go to LOCAL broker only
    if "local" in publishers:
        internal_messages = [
            {"type": "heartbeat", "process_id": "worker-1", "timestamp": time.time()},
            {"type": "sensor_reading", "sensor_id": "temp-01", "value": 23.5},
            {"type": "debug_log", "level": "info", "message": "Process initialized"}
        ]
        
        print("Internal communications (LOCAL broker):")
        for msg in internal_messages:
            topic = f"{cfg.mqtt_configs['local'].base_topic}/internal/{msg['type']}"
            payload = json.dumps(msg)
            result = publishers["local"].publish_json(topic, payload)
            status = "✓" if result.rc == 0 else "✗"
            print(f"  {status} {topic}")
    
    # Status messages go to PUBLIC brokers for external visibility
    public_brokers = [b for b in publishers.keys() if b != "local"]
    if public_brokers:
        status_message = {
            "system_status": "running",
            "uptime_seconds": 3600,
            "active_processes": 5,
            "timestamp": time.time()
        }
        
        print(f"\nStatus updates (PUBLIC brokers):")
        for broker_name in public_brokers:
            broker_cfg = cfg.mqtt_configs[broker_name]
            topic = f"{broker_cfg.base_topic}/status/system"
            payload = json.dumps(status_message)
            result = publishers[broker_name].publish_json(topic, payload)
            status = "✓" if result.rc == 0 else "✗"
            print(f"  {status} {broker_name}: {topic}")

In [ ]:
# Clean up: disconnect from all brokers
for profile_name, connector in connectors.items():
    connector.disconnect()
    print(f"Disconnected from {profile_name}")

print("\nAll connections closed")